In [ ]:
from google.colab import drive
import os

#unzip Full dataset
# 1. Mount Google Drive
drive.mount('/content/drive')
print("Step 1 finished")

# 2. Define the path to your new zip file in Drive
drive_zip_path = "/content/drive/MyDrive/Pokemon Labelled Classification Images.zip"
local_zip_path = "/content/Pokemon_Full_Dataset.zip"
print("Step 2 finished")

# 3. Copy the zip to the local Colab disk for faster access
!cp "$drive_zip_path" "$local_zip_path"
print("Step 3 Finished")

# 4. Unzip the dataset
!unzip -q "$local_zip_path" -d /content
print("Step 4 Finished")

# 5. Verification
num_classes = len(os.listdir('/content/Pokemon Labelled Classification Images'))
print(f"Dataset extracted. Found {num_classes} folders (classes).")
print("Step 5 Finished")


Mounted at /content/drive
Step 1 finished
Step 2 finished
Step 3 Finished
Step 4 Finished
Dataset extracted. Found 1070 folders (classes).
Step 5 Finished


In [ ]:
# Ensure that the data of all the pokemon can be accessed
from google.colab import drive
import os

data_dir = '/content/Pokemon Labelled Classification Images'
classes = sorted(os.listdir(data_dir))

# Remove system files if they exist
if '.DS_Store' in classes:
    classes.remove('.DS_Store')

# Save the index mapping to a text file
with open('index.txt', 'w') as f:
    for i, class_name in enumerate(classes):
        line = f"{i + 1}: {class_name}\n"
        f.write(line)

print(f"Successfully saved {len(classes)} classes to index.txt")



In [ ]:
import shutil
#unzip sugimori artwork
# 2. Define the path to your new zip file in Drive
drive_zip_path = "/content/drive/MyDrive/Pokedex Image Dataset.zip"
local_zip_path = "/content/Pokemon_Sugimori_Dataset.zip"
print("Step 2 finished")

# 3. Copy the zip to the local Colab disk for faster access
!cp "$drive_zip_path" "$local_zip_path"
print("Step 3 Finished")

# 4. Unzip the dataset
!unzip -q "$local_zip_path" -d /content
print("Step 4 Finished")

# 5. Verification
num_classes = len(os.listdir('/content/Pokedex Image Dataset'))
print(f"Dataset extracted. Found {num_classes} folders (classes).")
print("Step 5 Finished")

# Define your paths
source_dir = "Pokedex Image Dataset"
target_base_dir = "Pokemon Labelled Classification Images"

def move_to_existing_folders():
    # Verify paths exist
    if not os.path.exists(source_dir) or not os.path.exists(target_base_dir):
        print("Error: One of the directories does not exist. Check your paths.")
        return

    moved_count = 0
    missing_folders = set()

    for filename in os.listdir(source_dir):
        if filename.lower().endswith('.png'):
            # Strip the .png to get the Pokemon name/ID
            pokemon_name = os.path.splitext(filename)[0]

            # Construct the path to the expected destination folder
            destination_folder = os.path.join(target_base_dir, pokemon_name)

            # Check if the destination folder actually exists
            if os.path.exists(destination_folder):
                source_path = os.path.join(source_dir, filename)
                destination_path = os.path.join(destination_folder, filename)

                shutil.move(source_path, destination_path)
                moved_count += 1
            else:
                missing_folders.add(pokemon_name)

    print(f"Success! Moved {moved_count} files.")

    if missing_folders:
        print(f"Warning: {len(missing_folders)} files had no matching folder in the target directory.")
        # Optional: print(missing_folders) to see which ones failed

move_to_existing_folders()

Step 2 finished
Step 3 Finished
replace /content/__MACOSX/._Pokedex Image Dataset? [y]es, [n]o, [A]ll, [N]one, [r]ename: Step 4 Finished
Dataset extracted. Found 1 folders (classes).
Step 5 Finished
Success! Moved 0 files.


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, datasets
import torch.optim as optim
import os

torch.backends.cuda.matmul.allow_tf32 = True

torch.backends.cudnn.allow_tf32 = True

# 1. Load ResNet-50 instead of AlexNet
model = models.resnet50(weights='ResNet50_Weights.DEFAULT')

# 2. Modify the last layer
# In ResNet, the final layer is called 'fc'.
# We dynamically get the 'in_features' (which is 2048 for ResNet-50)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1069)

# 3. Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 4. Automatically calculate weights for 1069 classes
dataset = datasets.ImageFolder('/content/Pokemon Labelled Classification Images')
class_counts = torch.zeros(1069)
for _, label in dataset.samples:
    class_counts[label] += 1
print(len(class_counts))

# Calculate inverse weights to handle imbalance
weights = 1.0 / class_counts
weights = weights / weights.sum()
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

# 5. Optimizer stays similar
optimizer = optim.Adam(model.parameters(), lr=0.0001)

1069
1069


In [ ]:
#Splitting training, testing, and validation sets
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Requires 224x224 images
transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#Make sure that images aren't corrupted
from PIL import Image
import os
from PIL import PngImagePlugin

PngImagePlugin.MAX_TEXT_CHUNK = 100 * 1024 * 1024


def verify_images(folder_path):
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if not file.startswith('.'): # Skip .DS_Store
                file_path = os.path.join(root, file)
                try:
                    with Image.open(file_path) as img:
                        img.verify() # Check if the image is corrupted
                except (IOError, SyntaxError, Image.UnidentifiedImageError, Image.DecompressionBombError):
                    print(f'Removing corrupted image: {file_path}')
                    os.remove(file_path)
verify_images(data_dir)

full_dataset = datasets.ImageFolder(data_dir, transform=transform)

#ensure that each class has at least 1 training, 1 validation, and 1 test image
import numpy as np
from torch.utils.data import Subset

# 1. Group all indices by their class/label
indices_by_class = {}
for idx, (_, label) in enumerate(full_dataset.samples):
    if label not in indices_by_class:
        indices_by_class[label] = []
    indices_by_class[label].append(idx)

train_indices = []
val_indices = []
test_indices = []
remaining_pool = []

# 2. Pick 3 random images from each class for the initial mandatory split
for label, indices in indices_by_class.items():
    # Shuffle indices within the class to ensure randomness
    np.random.shuffle(indices)

    # Assign the first 3 to Train, Val, and Test respectively
    train_indices.append(indices[0])
    val_indices.append(indices[1])
    test_indices.append(indices[2])

    # Put the rest of the images for this class into the remaining pool
    remaining_pool.extend(indices[3:])

# 3. Perform the 80/10/10 split on the remaining pool
np.random.shuffle(remaining_pool)
total_remaining = len(remaining_pool)

rem_train_size = int(0.8 * total_remaining)
rem_val_size = int(0.1 * total_remaining)

# Distribute remaining images
train_indices.extend(remaining_pool[:rem_train_size])
val_indices.extend(remaining_pool[rem_train_size : rem_train_size + rem_val_size])
test_indices.extend(remaining_pool[rem_train_size + rem_val_size:])

# 4. Create the final Subsets
train_data = Subset(full_dataset, train_indices)
val_data = Subset(full_dataset, val_indices)
test_data = Subset(full_dataset, test_indices)

print(f"Training size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")
print(f"Testing size: {len(test_data)}")
'''
# Calculate lengths for 80/10/10 split
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
print(f"Training size: {train_size}")
print(f"Validation size: {val_size}")
print(f"testing size: {test_size}")

train_data, val_data, test_data = random_split(full_dataset, [train_size, val_size, test_size])'''

# Create DataLoaders

train_loader = DataLoader(
    train_data,
    batch_size=256,    # Doubled from 64 to use the remaining GPU RAM
    shuffle=True,
    num_workers=12,     # Increased to use more System RAM/CPU
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True # Keeps workers active between epoch
)

# Validation Loader - Higher batch size is okay here!
val_loader = DataLoader(
    val_data,
    batch_size=512,    # Increased because no gradients are stored
    shuffle=False,     # DO NOT shuffle validation
    num_workers=8,
    pin_memory=True
)

# Test Loader - Match the validation settings
test_loader = DataLoader(
    test_data,
    batch_size=512,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)


In [ ]:
from tqdm.notebook import tqdm

num_epochs = 25
best_val_acc = 0.0

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", unit="batch")

    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_bar.set_postfix(loss=loss.item())

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    avg_train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct_train / total_train

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", unit="batch", leave=False)

    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct_val / total_val

    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')

    # --- Save Best Model ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_pokemon_model.pth')
        print("Model saved!")



In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
print(test_data)

# 1. Testing Phase with Progress Bar
model.eval()
test_correct = 0
test_total = 0
all_preds = []
all_labels = []

test_bar = tqdm(test_loader, desc="Final Testing", unit="batch")

with torch.no_grad():
    for inputs, labels in test_bar:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # Live update of accuracy in the progress bar
        test_bar.set_postfix(acc=f"{100 * test_correct / test_total:.2f}%")

test_accuracy = 100 * test_correct / test_total
print(f"\nFinal Test Accuracy on {len(classes)} Classes: {test_accuracy:.2f}%")

# 2. Confusion Matrix Logic for 1,000+ Classes
cm = confusion_matrix(all_labels, all_preds)

np.save('full_confusion_matrix.npy', cm)

plt.figure(figsize=(15, 12))
sns.heatmap(cm[:25, :25], annot=True, fmt='d', cmap='Blues',
            xticklabels=classes[:25], yticklabels=classes[:25])

plt.title('ResNet-50 Results: Confusion Matrix (Subset of First 25 Classes)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('confusion_matrix_subset.png', bbox_inches='tight')
plt.show()

# 3. Generate Classification Report (Precision, Recall, F1)
report = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
import pandas as pd
report_df = pd.DataFrame(report).transpose()
report_df.to_csv('classification_report.csv')

print("Results saved: 'confusion_matrix_subset.png' and 'classification_report.csv'")
